In [2]:
import gradio as gr
import pickle

# 1. Load the pre-trained model and vectorizer from disk
with open('bernoulli_model.pkl', 'rb') as model_file:
    ber = pickle.load(model_file)

with open('count_vectorizer.pkl', 'rb') as cv_file:
    cv = pickle.load(cv_file)


# 2. Prediction function using the loaded model
def predict_sentiment(review_text):
    if not review_text.strip():
        return "⚠️ Please enter a review to analyze.", "", None
    
    # Preprocessing
    x = clean_html(review_text)
    x = lower_conv(x)
    x = remove_special(x)
    x = clean_stopwords(x)
    x = clean_stem(x)
    x = join_to_string(x)
    
    # Vectorize using loaded CountVectorizer
    x_vector = cv.transform([x]).toarray()
    
    # Predict using loaded BernoulliNB
    pred = ber.predict(x_vector)[0]
    probs = ber.predict_proba(x_vector)[0]
    
    prob_dict = {
        "Positive": float(probs[1]),
        "Negative": float(probs[0])
    }
    
    result_badge = "🎉 POSITIVE SENTIMENT" if pred == 1 else "🚨 NEGATIVE SENTIMENT"
        
    return result_badge, x, prob_dict


# 3. Build UI Blocks
with gr.Blocks(theme=gr.themes.Soft(), title="Movie Review Sentiment Analyzer") as demo:
    
    gr.Markdown(
        """
        # 🎬 Movie Review Sentiment Analyzer
        *Powered by a pre-trained Bernoulli Naive Bayes Model*
        """
    )
    
    with gr.Row():
        # Left Column: Inputs
        with gr.Column(scale=2):
            review_input = gr.Textbox(
                lines=5,
                placeholder="Type or paste your movie review here...",
                label="📝 Review Text"
            )
            
            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear")
                submit_btn = gr.Button("🔍 Analyze Sentiment", variant="primary")
                
            gr.Examples(
                examples=[
                    ["The acting was fantastic and the plot kept me engaged from start to finish!"],
                    ["Terrible movie. A complete waste of time and money."],
                    ["It had some decent visual effects, but overall the storyline was slow and boring."]
                ],
                inputs=review_input,
                label="💡 Click an Example Review"
            )

        # Right Column: Results
        with gr.Column(scale=2):
            result_output = gr.Textbox(label="🏷️ Verdict", interactive=False)
            prob_output = gr.Label(label="📊 Confidence Score")
            cleaned_preview = gr.Textbox(label="🧹 Cleaned Text Preview", lines=2, interactive=False)

    # Event handlers
    submit_btn.click(
        fn=predict_sentiment,
        inputs=review_input,
        outputs=[result_output, cleaned_preview, prob_output]
    )
    
    clear_btn.click(
        fn=lambda: ("", "", None),
        outputs=[review_input, result_output, prob_output]
    )

demo.launch(inbrowser=True)


/var/folders/py/_l9sj33x5cv9b64dhj56sqq00000gn/T/ipykernel_60432/2015779789.py:43: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Movie Review Sentiment Analyzer") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
